# Machine-learning features: the NPZ bundle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mgoldey/ferric/blob/main/examples/notebooks/05_ml_features.ipynb)

A `pdep-rpa` run exports per-molecule and per-atom properties (electrostatic potential, charges, polarisabilities, C6) to a compressed NumPy archive. This notebook builds a small feature table from three molecules.

Runs on a Colab CPU runtime in about a minute. Every number printed below is also asserted, so a cell that runs without an error has reproduced the reference value.

In [ ]:
%pip install -q --pre ferric py3Dmol
import os

os.environ.setdefault(
    "OPENBLAS_NUM_THREADS", "1"
)  # set before the first `import ferric`

In [ ]:
import os
import shutil
import subprocess
import numpy as np
import ferric

assert shutil.which("ferric"), "the `ferric` command is installed with the wheel"
print(ferric.__build__)

The bundle comes from the command-line front end, which reads a TOML input (every key is in the [input reference](https://mgoldey.github.io/ferric/reference/input.html)). The schema and units are in [Export](https://mgoldey.github.io/ferric/using/export.html): coordinates in Bohr, ESP in Hartree/e, polarisabilities in Bohr^3, charges in e.

In [ ]:
MOLECULES = {
    "water": """3
water
O   0.000000   0.000000   0.117790
H   0.000000   0.755453  -0.471161
H   0.000000  -0.755453  -0.471161
""",
    "ammonia": """4
ammonia
N   0.000000   0.000000   0.000000
H   0.937330   0.000000  -0.381520
H  -0.468670   0.811740  -0.381520
H  -0.468670  -0.811740  -0.381520
""",
    "methane": """5
methane
C   0.000000   0.000000   0.000000
H   0.628740   0.628740   0.628740
H   0.628740  -0.628740  -0.628740
H  -0.628740   0.628740  -0.628740
H  -0.628740  -0.628740   0.628740
""",
}

TOML = """[molecule]
xyz = "{name}.xyz"

[basis]
name = "cc-pvdz"

[method]
kind = "pdep-rpa"
task = "energy"

[rpa]
auxbasis = "cc-pvdz-ri"
n_quad = 20
trunc_thresh = 1e-4
export_npz = "{name}.npz"
compute_esp = true
compute_polarizability = true
"""

os.makedirs("features", exist_ok=True)
env = dict(os.environ, OPENBLAS_NUM_THREADS="1")
for name, xyz in MOLECULES.items():
    with open(f"features/{name}.xyz", "w") as f:
        f.write(xyz)
    with open(f"features/{name}.toml", "w") as f:
        f.write(TOML.format(name=name))
    run = subprocess.run(
        ["ferric", f"{name}.toml"],
        cwd="features",
        env=env,
        capture_output=True,
        text=True,
    )
    assert run.returncode == 0, run.stderr
    print(
        name,
        "->",
        [ln for ln in run.stdout.splitlines() if ln.startswith("Total (RHF+RPA)")][0],
    )

## Read a bundle

In [ ]:
d = np.load("features/water.npz")
for key in (
    "coords",
    "atomic_numbers",
    "esp_atoms",
    "lowdin_charges",
    "chelpg_charges",
    "alpha_atomic",
    "alpha_tensor",
    "c6_molecular_iso",
):
    print(f"{key:18s}", d[key].shape)

## A feature table

Per atom: the Loewdin charge, the electrostatic potential at the nucleus and the isotropic part of the Becke-partitioned (Krishtal) polarisability. The partitioned polarisability leaves out charge transfer, which the bundle keeps separately in `alpha_ct`, so the molecular tensor is their sum.

In [ ]:
print(
    f"{'molecule':8s} {'atom':>4s} {'Z':>2s} {'q_Loewdin':>10s} {'ESP(nuc)':>10s} {'alpha_iso':>10s}"
)
for name in MOLECULES:
    d = np.load(f"features/{name}.npz")
    alpha_iso = np.trace(d["alpha_atomic"], axis1=1, axis2=2) / 3
    for i, z in enumerate(d["atomic_numbers"]):
        print(
            f"{name:8s} {i:4d} {z:2d} {d['lowdin_charges'][i]:10.4f} {d['esp_atoms'][i]:10.3f} {alpha_iso[i]:10.4f}"
        )

    # Documented identities: neutral molecules have zero total charge, and the molecular polarisability
    # is the atomic parts plus the charge-transfer remainder.
    assert abs(d["lowdin_charges"].sum()) < 1e-6
    assert (
        np.abs(d["alpha_ct"] + d["alpha_atomic"].sum(axis=0) - d["alpha_tensor"]).max()
        < 1e-10
    )
    print(
        f"{name:8s} molecular alpha_iso = {np.trace(d['alpha_tensor']) / 3:.3f} Bohr^3, "
        f"C6 = {d['c6_molecular_iso'][0]:.2f} a.u.\n"
    )

The arrays stack directly into a dataset: `np.stack` the per-atom columns over molecules (pad to the largest atom count), or keep one `.npz` per molecule and load them lazily.